<b><font size="6" color="#E8800A">Week 4a · Feature work and dimensionality reduction before selection</font></b><br>
<b><font size="4">Engineered features, and what selection keeps and costs</font></b><br>

The cleaning recipe produces `champions.csv`: 4,000 competition
entries whose preparation is documented.

Engineer features before selection, because a selector cannot retain a feature
that does not yet exist. Then compare filters, wrappers and embedded methods by their held-out score and
the columns they keep.

<div class="alert alert-block alert-info">

# TOC<a class="anchor" id="toc"></a>
* [<font color='#E8800A'>The Pre-Processing Regime</font>](#regime)
* [<font color='#E8800A'>Encoding and scaling</font>](#encoding)
* [<font color='#E8800A'>Feature extraction and engineering</font>](#transform)
* [<font color='#E8800A'>Combination features</font>](#combination)
* [<font color='#E8800A'>Dimensionality reduction</font>](#svd)
* [<font color='#E8800A'>Feature selection I: Filter methods</font>](#filter)
* [<font color='#E8800A'>Feature selection II: Wrapper methods</font>](#wrapper)
* [<font color='#E8800A'>Feature selection III: Embedded methods</font>](#embedded)
* [<font color='#E8800A'>Combining strategies</font>](#combined)
* [<font color='#E8800A'>So what is selection for?</font>](#synthesis)
* [<font color='#E8800A'>Key takeaways</font>](#takeaways)
* [<font color='#E8800A'>References</font>](#references)

</div>

# <font color='#E8800A'>The Pre-Processing Regime</font> <a class="anchor" id="regime"></a>
[Back to TOC](#toc)

The cleaning log carries five decisions, and this notebook applies three
of them: the dtype of the six boolean columns, the column roles with their median
or mode fills, and `log1p` on eleven training-volume columns, the logged outlier
treatment. The log stores rules, not fitted values. Each split learns its own
medians, modes, categories and scaling statistics from its training rows; fitting
them before the split would leak held-out information.

This notebook adds the encoding and scaler chosen by a sixteen-regime benchmark.
Together, those entries define the preprocessing recipe used below.

Every result below uses one protocol. Twenty repeated 80/20 splits are drawn
at the start. Every learned step, from filling a gap to
choosing a feature, is fitted on a split's training 80% and scored on its
held-out 20%, and every comparison is made split by split.

__Step 1:__ Import the libraries and set up the notebook. The setup
installs `category_encoders` only when it is missing.

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec("category_encoders") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "category-encoders"]
    )

# `course_helpers.py` sits beside the notebook, and Python does not search
# that folder on its own.
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

# Standard library: `Counter` tallies how often a column is selected,
# `inspect` reads a selector's signature, `textwrap` folds printed text.
import inspect
import textwrap
import warnings
from collections import Counter

# Arrays, frames and plots.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
sns.set_theme(style="whitegrid")

# Encoders for the categorical columns, and the two association statistics the
# typed filter scores a source column with.
from category_encoders import CountEncoder, TargetEncoder
from scipy.stats import chi2_contingency, pointbiserialr

# Feature extraction first, then the selectors this week compares.
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_selection import (
    RFE,
    RFECV,
    SelectFromModel,
    SelectKBest,
    SequentialFeatureSelector,
    VarianceThreshold,
)

# The imputer that fills each split, the model every technique is scored
# through, the splits it is scored on, and the score reported for it.
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedShuffleSplit

# The scalers and encoders refitted inside every training split, and the
# warning the solver raises when it stops at `max_iter`.
from sklearn.preprocessing import (
    MinMaxScaler,
    OneHotEncoder,
    OrdinalEncoder,
    RobustScaler,
    StandardScaler,
)
from sklearn.exceptions import ConvergenceWarning

# The course's shared cleaning log and plot colours.
from course_helpers import CleaningLog, PLOT_BLUE, PLOT_ORANGE

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)  # For reproducibility
FIGSIZE = (6, 5)

# Convergence, one-hot and pandas deprecation warnings are silenced once here.
# The source of each is known and expected.
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

__Step 2:__ Open the cleaning log. Each `CleaningLog` entry records the
affected columns, the decision, its reason, the number of changed rows, and a
machine-readable `carries` value.

In [ ]:
recipe_log = CleaningLog.load("../../logs/champions_cleaning_log.json")

print(f"{len(recipe_log)} decisions inherited from Week 3\n")
for decision in recipe_log.steps:
    carried = ", ".join(sorted(decision.carries))
    print(f"  {decision.column:28s} {decision.action}")
    print(f"  {'':28s} carries -> {carried}\n")

__Step 3:__ Load the cleaned spine with the dtypes the log records, and separate
the target from the features. `RecordID` becomes the row index, which identifies
a row without making it a feature. `Athlete Id` is the identifier used for
deduplication, so it is not a model feature either.

Run this notebook from its own folder inside the course
repository: that is what makes the `data/` paths below work. If you
downloaded this file on its own from Moodle, move it into the repository before
you run it.

In [ ]:
# A CSV stores no dtypes, so the six booleans, which carry gaps, would come
# back as `object`. The log's cast names the dtype each one should have.
cast = recipe_log.plan("6 boolean columns")["cast"]
champions = pd.read_csv("../../data/interim/champions.csv", dtype=cast)
champions = champions.set_index("RecordID")

X = champions.drop(columns=["Outcome", "Athlete Id"])
y = champions["Outcome"]

# Week 3's repeated 80/20 design
splitter = StratifiedShuffleSplit(n_splits=20, test_size=0.20,
                                  random_state=RANDOM_STATE)
splits = list(splitter.split(X, y))

print(f"{X.shape[0]:,} rows x {X.shape[1]} source columns")
print(f"{X.isna().sum().sum():,} gaps in the file,"
      f" filled per split from that split's own training rows")
print(f"win rate {y.mean():.4f}")
print(f"{len(splits)} stratified 80/20 splits,"
      f" {len(splits[0][1])} rows held out in each")

__Step 4:__ Read the recipe from the log rather than retyping it. `plan` finds a
labelled step and returns its `carries`. The twenty-six-column step names both
column blocks and their fills, and the outlier step names the columns `log1p`
applies to. Each logged fill becomes the scikit-learn imputer it names. Encoding
and scaling stay open until the benchmark below measures them together.

In [ ]:
fill_step = recipe_log.plan("26 columns")
numeric, categorical = fill_step["numeric"], fill_step["categorical"]
fill = fill_step["fill"]
log1p_columns = recipe_log.plan("11 training-volume columns")["columns"]

# The imputer each logged fill names. Every split refits it on its training rows.
imputers = {"median": SimpleImputer(strategy="median"),
            "mode": SimpleImputer(strategy="most_frequent")}
recipe = {"numeric": numeric, "categorical": categorical,
          "fill": {"numeric": imputers[fill["numeric"]],
                   "categorical": imputers[fill["categorical"]]},
          "log1p": log1p_columns}

print(f"{len(numeric)} numeric, {len(categorical)} categorical")
print(f"numeric gaps take the training {fill['numeric']},"
      f" categorical gaps the training {fill['categorical']}")
print(f"log1p on {len(log1p_columns)} of the {len(numeric)} numeric columns;"
      f" the other {len(numeric) - len(log1p_columns)} are"
      f" {[c for c in numeric if c not in log1p_columns]}")

<div class="alert alert-block alert-info">

**A stored rule is not a stored fit.** Each training split supplies
its own medians, modes, encoder categories and scaling statistics. Only `log1p`
is stateless. The benchmark rechecks all four scaler regimes with every encoder.

</div>

__Step 5:__ Define the fold mechanics before reading any result. `fill_missing`
fills each block of columns with its imputer. `fold_matrices` runs the recipe on
one split in a fixed order: fill, encode, `log1p`, scale. Every fit takes the
split's training rows, and the held-out rows only reach a `transform`.

In [ ]:
def fill_missing(train, test, plan):
    """Fill every missing value with imputers fitted on `train` alone.

    `plan` is a list of (columns, imputer) pairs, and any scikit-learn imputer
    works: each one is fitted on the training rows of its own columns and fills
    those columns in both halves, so the test rows never help compute a fill.
    """
    train, test = train.copy(), test.copy()
    for columns, imputer in plan:
        # scikit-learn reads np.nan as a gap but not pd.NA, the gap of a
        # nullable column such as the booleans, so every gap becomes np.nan.
        blocks = [half[columns].astype(object).fillna(np.nan).infer_objects()
                  for half in (train, test)]
        imputer.fit(blocks[0])
        train[columns], test[columns] = [imputer.transform(block) for block in blocks]
    return train, test


# One encoder and one scaler per candidate, refitted inside every split. One-hot
# always drops one level per column.
encoders = {
    "one-hot": OneHotEncoder(handle_unknown="ignore", drop="first",
                             sparse_output=False),
    "ordinal": OrdinalEncoder(handle_unknown="use_encoded_value",
                              unknown_value=-1),
    "count": CountEncoder(normalize=True, handle_unknown=0,
                          handle_missing="value"),
    "target": TargetEncoder(handle_unknown="value", handle_missing="value"),
}
scalers = {"none": None, "standard": StandardScaler,
           "min-max": MinMaxScaler, "robust": RobustScaler}


def fold_matrices(train, test, y_train, recipe):
    """Return (A_train, A_test, names) for ONE split, every step fitted on `train`.

    The categorical block comes first in the matrix and the numeric block
    second. That width is not constant across splits, because a level absent
    from a training split contributes no column to it.
    """
    numeric, categorical = recipe["numeric"], recipe["categorical"]

    # 1. FILL first, because the spine ships with its gaps open.
    train, test = fill_missing(train, test,
                               [(numeric, recipe["fill"]["numeric"]),
                                (categorical, recipe["fill"]["categorical"])])

    # 2. ENCODE. Every encoder is handed the training outcome, and only target
    #    encoding reads it.
    encoder = encoders[recipe["encoding"]]
    cat_train = encoder.fit_transform(train[categorical], y_train)
    cat_test = encoder.transform(test[categorical])
    names = list(encoder.get_feature_names_out(categorical)) + numeric

    # 3. log1p on the logged columns. It fits nothing.
    num_train = train[numeric].to_numpy(dtype=float).copy()
    num_test = test[numeric].to_numpy(dtype=float).copy()
    where = [numeric.index(column) for column in recipe["log1p"]]
    num_train[:, where] = np.log1p(num_train[:, where])
    num_test[:, where] = np.log1p(num_test[:, where])

    # 4. SCALE, with the centre and spread of the training rows. Ordinal, count
    #    and target encoding turn a level into a number, so the scaler applies
    #    to them too; one-hot indicators stay 0/1.
    make = scalers[recipe["scaler"]]
    if make is not None:
        fitted = make().fit(num_train)
        num_train, num_test = fitted.transform(num_train), fitted.transform(num_test)
        if recipe["encoding"] != "one-hot":
            fitted = make().fit(cat_train)
            cat_train, cat_test = fitted.transform(cat_train), fitted.transform(cat_test)

    return (np.hstack([cat_train, num_train]),
            np.hstack([cat_test, num_test]),
            np.asarray(names))

<div class="alert alert-block alert-info">

These helpers are written in the notebook rather than imported,
so every fit boundary stays visible at the point it matters. Read that as a
teaching arrangement and not as a recommendation: once the helpers multiply, a
notebook is the wrong home for them. From Week 5 onward this logic is
imported from `preprocessing.py` beside the notebook, and the split is worth
making in your own work: one module per job, preprocessing in one and
evaluation in another.

</div>

# <font color='#E8800A'>Encoding and scaling</font> <a class="anchor" id="encoding"></a>
[Back to TOC](#toc)

A model sees a matrix of numbers, so the twelve categorical columns
need a representation. One-hot, ordinal, count and target encoding make different
assumptions and different matrix widths.

<div class="alert alert-block alert-info">

**The candidates do not make the same claim.** One-hot keeps a
separate indicator for each non-reference level and always uses `drop="first"`.
Ordinal is compact but imposes an arbitrary numeric order. Count is also compact
and replaces a level with its frequency in the training split. Target encoding
uses the outcome, so its fit receives only the training 80%; the held-out 20%
never contributes to its category means.

**A number an encoder made is still a number.** Ordinal, count and target
encoding each replace a level with a numeric score, and a penalised model reads
that score on whatever scale the encoder produced. So the scaler under test is
applied to those scores as well, fitted on the same training rows as the
measurements. One-hot indicators are left as 0/1: their spread only records how
common a level is.

</div>

__Step 6:__ `heldout_f1` scores a recipe on the twenty splits, and every result in
this notebook comes from it. It prepares each split with `fold_matrices` and fits
a logistic regression on the training rows. It returns one row per split: the
number of columns the model saw, its F1 on the held-out rows, those columns and
the fitted selector. Two optional steps sit between the preparation and the
model, each fitted on the training matrix alone. `reduce` compresses the matrix
into components, and `select` keeps some of its columns.

In [ ]:
def heldout_f1(recipe, frame=X, reduce=None, select=None):
    """Held-out F1 on every split, one row per split.

    `reduce(n)` builds a reducer of n components, refitted with the fewest
    components that keep 80% of the split's variance. `select(split)` builds a
    selector for that split. The model sees the columns `keep` marks: all of
    them, unless a selector keeps fewer.
    """
    rows = []
    for split, (train_index, test_index) in enumerate(splits):
        y_train = y.iloc[train_index]
        A_train, A_test, names = fold_matrices(
            frame.iloc[train_index], frame.iloc[test_index], y_train, recipe)
        if reduce is not None:
            probe = reduce(A_train.shape[1] - 1).fit(A_train)
            k = int(np.searchsorted(probe.explained_variance_ratio_.cumsum(), 0.80) + 1)
            reducer = reduce(k).fit(A_train)
            A_train, A_test = reducer.transform(A_train), reducer.transform(A_test)
            names = np.asarray([f"component {i}" for i in range(1, k + 1)])
        selector, keep = None, np.ones(len(names), dtype=bool)
        if select is not None:
            # The selector is handed the matrix WITH its column names, because a
            # filter that judges a categorical as a whole has to know which
            # dummies came from it.
            selector = select(split).fit(pd.DataFrame(A_train, columns=names),
                                         y_train)
            keep = selector.get_support()
        model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
        model.fit(A_train[:, keep], y_train)
        rows.append({"n": int(keep.sum()),
                     "F1": f1_score(y.iloc[test_index], model.predict(A_test[:, keep])),
                     "kept": list(names[keep]), "selector": selector})
    return pd.DataFrame(rows)

__Step 7:__ Cross every encoder with every scaler: sixteen recipes, each scored on
the same twenty splits. Choose by mean held-out F1, and judge each gap against
its standard error.

In [ ]:
encoding_runs = {
    (encoding, scaler): heldout_f1({**recipe, "encoding": encoding, "scaler": scaler})
    for encoding in encoders for scaler in scalers
}
encoding_board = pd.DataFrame.from_dict(
    {pair: {"F1": run["F1"].mean(), "F1 SEM": run["F1"].sem()}
     for pair, run in encoding_runs.items()},
    orient="index",
).rename_axis(["encoding", "scaler"])

# idxmax returns the (encoding, scaler) label of the best row, not its score.
selected_encoding, selected_scaler = encoding_board["F1"].idxmax()
recipe = {**recipe, "encoding": selected_encoding, "scaler": selected_scaler}
# The selected recipe's own run: every comparison below is paired against it.
reference = encoding_runs[selected_encoding, selected_scaler]

print(encoding_board.round(4).to_string())
print(f"selected pair: {selected_encoding} + {selected_scaler}"
      " (highest held-out F1)")

__Step 8:__ **Exercise.** Encode with `drop="first"` and count the features.
Then check the widths across the repeated training splits, and explain the difference.

In [ ]:
widths = []
for train_index, test_index in splits:
    # 1. encode this split from the RAW rows; one encoder fitted outside the
    #    loop would have seen every category, the test rows' included
    A_train, _, _ = fold_matrices(...)  # <-- CODE HERE
    # 2. record the width this split produced
    widths.append(...)  # <-- CODE HERE

# 3. the whole frame, encoded once on the rows that carry no gaps
complete = X.dropna()
whole_frame = ...  # <-- CODE HERE

print("whole-frame width:", ...)  # <-- CODE HERE
print("per-split widths :", widths)
# 4. count the rows carrying the level that accounts for the gap
print("'No coach' True rows in all 4,000:", ...)  # <-- CODE HERE

<div class="alert alert-block alert-warning">

**The encoded frame is 48 or 49 columns wide depending on the
split.** `No coach` is `True` for exactly **one** athlete in 4,000. Whenever that
single row lands in the held-out 20%, the training split sees only one level of
`No coach`, and `drop="first"` drops it, so the column does not exist at all for
that split.

</div>

# <font color='#E8800A'>Feature extraction and engineering</font> <a class="anchor" id="transform"></a>
[Back to TOC](#toc)

Models often need columns transformed, combined or encoded before
selection. The recipe's transform is `log1p` on the eleven training-volume
columns, which every split applies as the log records.

<div class="alert alert-block alert-info">

Other common transforms differ in strength and interpretation:

- **`np.sqrt`** is a milder, stateless compression of a right tail.
- [**`PowerTransformer`**](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PowerTransformer.html)
  estimates a normalising power. Yeo-Johnson accepts zero and negative values;
  Box-Cox requires strictly positive values.
- [**`QuantileTransformer`**](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.QuantileTransformer.html)
  maps ranks to a chosen distribution, changing the original distances.
- [**`KBinsDiscretizer`**](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.KBinsDiscretizer.html)
  replaces a measurement with ordered bands.

Unlike `log1p`, the last three are **fitted** and must learn their parameters
from training rows. Apply them to raw numeric columns, never the encoded matrix.
A 0/1 column cannot be de-skewed: mapping it to 0 and 0.693 only rescales it.

</div>

# <font color='#E8800A'>Combination features</font> <a class="anchor" id="combination"></a>
[Back to TOC](#toc)

A new feature can come from more than one column at once. The eleven
training-volume columns are all minutes; their sum is a single number for how much
this athlete trained.

__Step 9:__ `paired_delta` reads two runs split by split and returns the mean
difference, the standard error of that mean, how many splits the first run won,
and the verdict at two standard errors. Every comparison below goes through it.

In [ ]:
def paired_delta(scores, reference):
    """Mean split-by-split difference from `reference`, its SEM, wins and verdict."""
    delta = np.asarray(scores) - np.asarray(reference)
    sem = delta.std(ddof=1) / np.sqrt(len(delta))
    # Two standard errors separate a difference from noise; a delta that is
    # zero on every split is no difference at all.
    verdict = ("identical on every split" if not delta.any()
               else "inside the noise" if abs(delta.mean()) <= 2 * sem
               else "better" if delta.mean() > 0 else "worse")
    return {"dF1": delta.mean(), "SEM": sem,
            "wins": f"{int((delta > 0).sum())}/{len(delta)}", "verdict": verdict}

__Step 10:__ **Exercise.** `total_training` is the sum of the eleven
training-volume columns. Does one column that adds up the training load carry
what the eleven carry separately? Test it two ways against `reference`, the
recipe's own run: *added* beside the eleven, and *replacing* them. Both arms log
`total_training`, as the recipe logs its parts.

Both arms score `X_added`, the frame with `total_training` beside the
columns it sums, and differ only in the numeric columns their recipe hands the
model.

| arm | numeric columns the model sees | columns logged | compared with |
|---|---|---|---|
| added | the fourteen and `total_training` | the eleven and `total_training` | `reference` |
| REPLACING the eleven | the other three and `total_training` | `total_training` | `reference` |

In [ ]:
# 1. the sum of the eleven training-volume columns, beside the columns it sums
X_added = ...  # <-- CODE HERE

# 2. two recipes: the sum added to the numeric columns, and the sum standing in
#    for the eleven
added = {**recipe, "numeric": ...,  # <-- CODE HERE
         "log1p": ...}  # <-- CODE HERE
replaced = {**recipe,
            "numeric": ...,  # <-- CODE HERE
            "log1p": ...}  # <-- CODE HERE

# 3. each arm paired against the recipe's own run, on the same twenty splits
arms = pd.DataFrame({
    "added": ...,  # <-- CODE HERE
    "REPLACING the eleven": ...,  # <-- CODE HERE
}).T
print(arms.to_string(formatters={"dF1": "{:+.4f}".format, "SEM": "{:.4f}".format}))

<div class="alert alert-block alert-success">

Added beside its eleven parts, `total_training` moves held-out F1 by
**+0.0019**, inside the noise. Replacing the parts with their sum costs
**-0.0069**, so the sum is not a sufficient statistic for its parts.

</div>

# <font color='#E8800A'>Dimensionality reduction</font> <a class="anchor" id="svd"></a>
[Back to TOC](#toc)

Dimensionality reduction compresses the original columns into a
smaller set of new features. Selection instead keeps named columns and drops the
rest. That difference determines what the fitted model can still explain.

<div class="alert alert-block alert-info">

These methods solve different compression problems:

- [**Principal Component Analysis (`PCA`)**](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)
  keeps orthogonal directions with the greatest spread. It is a linear method
  computed from the singular value decomposition of a centred matrix.
- [**`TruncatedSVD`**](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html)
  avoids centring, so sparse inputs such as text remain sparse.
- [**`NMF`**](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.NMF.html)
  builds non-negative parts, so components can be read as ingredients.
- [**`FeatureAgglomeration`**](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.FeatureAgglomeration.html)
  averages groups of similar columns.
- [**Random projection**](https://scikit-learn.org/stable/modules/random_projection.html)
  uses a random matrix to preserve distances approximately at low cost.
- [**`TSNE`**](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html)
  and [UMAP](https://umap-learn.readthedocs.io/) are non-linear maps for *looking*
  at data in two dimensions, not model preprocessing.

All learn from data and therefore belong inside the training split.

</div>

__Step 11:__ See the idea on two columns before applying it to forty-nine.
`Train bf competition` and `Recovery` move together, so most of what they say
lies along one direction.

In [ ]:
# Split 0's training rows, prepared by the recipe: the matrix the model sees.
first_index, _ = splits[0]
first_matrix, _, first_names = fold_matrices(
    X.iloc[first_index], X.iloc[first_index], y.iloc[first_index], recipe)
first_y = y.iloc[first_index]

# Two of its columns, already logged and scaled by the recipe.
pair = ["Train bf competition", "Recovery"]
two = first_matrix[:, [list(first_names).index(column) for column in pair]]
pca_two = PCA(n_components=2).fit(two)
centre = two.mean(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].scatter(two[:, 0], two[:, 1], s=8, alpha=0.25, color=PLOT_BLUE)
for spread, direction, name in zip(np.sqrt(pca_two.explained_variance_),
                                   pca_two.components_, ("PC1", "PC2")):
    tip = centre + 2 * spread * direction
    axes[0].annotate("", xy=tip, xytext=centre,
                     arrowprops=dict(arrowstyle="-|>", color=PLOT_ORANGE, lw=2))
    axes[0].text(*tip, f"  {name}", color="0.2", fontsize=11)
axes[0].set_aspect("equal")
axes[0].set(xlabel=f"{pair[0]} (log1p, scaled)", ylabel=f"{pair[1]} (log1p, scaled)",
            title="Two columns, and the two directions PCA finds")

axes[1].hist(pca_two.transform(two)[:, 0], bins=40, color=PLOT_BLUE)
axes[1].set(xlabel="position along PC1", ylabel="athletes",
            title="The same rows as one number along PC1")
fig.tight_layout()
plt.show()

print(f"share of the spread along PC1: {pca_two.explained_variance_ratio_[0]:.4f}")

PC1 points along the cloud and PC2 across it. Projecting each row
onto PC1 keeps **88%** of the spread in one number. In forty-nine dimensions,
PCA repeats the search at right angles to the directions already found.

__Step 12:__ Fit `PCA` on the whole of that matrix. `explained` holds the share of
variance each component carries.

In [ ]:
pca = PCA().fit(first_matrix)
explained = pca.explained_variance_ratio_
print(f"{first_matrix.shape[1]} columns")

In [ ]:
# Nothing below is chosen from this chart. It shows how variance accumulates for
# both reducers, on one axis, because both curves are shares of the same total.
svd = TruncatedSVD(n_components=first_matrix.shape[1] - 1,
                   random_state=RANDOM_STATE).fit(first_matrix)
curves = pd.concat([
    pd.DataFrame({"components kept": np.arange(1, len(ratios) + 1),
                  "share of variance kept": ratios.cumsum(),
                  "reducer": name})
    for name, ratios in (("PCA", explained),
                         ("TruncatedSVD", svd.explained_variance_ratio_))
])

fig, ax = plt.subplots(figsize=FIGSIZE)
sns.lineplot(curves, x="components kept", y="share of variance kept",
             hue="reducer", palette=[PLOT_BLUE, PLOT_ORANGE],
             marker="o", markersize=3, ax=ax)
ax.axhline(0.80, color="grey", linewidth=1)
ax.set_ylim(0, 1.02)
ax.set(title="How variance accumulates, one component at a time")
ax.legend(title=None)
plt.show()

for name, ratios in (("PCA", explained),
                     ("TruncatedSVD", svd.explained_variance_ratio_)):
    reach = int(np.searchsorted(ratios.cumsum(), 0.80) + 1)
    print(f"{name:13s} first component {ratios[0]:.4f}"
          f"   components to reach 80%: {reach}")

__Step 13:__ **Exercise.** Report how many components reach 80%, 90% and 95% of the
variance, first from `explained`. Then read the same shares and the same three
counts out of `numpy.linalg.svd` on the centred matrix.

In [ ]:
# PCA is not the SVD; it is computed FROM it.
# 1. centre the columns, factor the centred matrix, and square the singular
#    values: their shares are the variance shares
_, S, _ = ...  # <-- CODE HERE
singular = ...  # <-- CODE HERE

for label, shares in (("PCA", explained), ("SVD", singular)):
    # 2. the count of components whose running total reaches each level
    needed = [...  # <-- CODE HERE
              for t in (0.80, 0.90, 0.95)]
    print(f"{label}  first three shares {np.round(shares[:3], 4).tolist()}"
          f"   components for 80% / 90% / 95%: {needed}")

__Step 14:__ A loading is the signed weight a source feature contributes to a
principal component. Its magnitude shows how strongly the feature contributes,
while its sign shows direction relative to the other weights. Read what the
first component is made of.

In [ ]:
loadings = pd.Series(pca.components_[0], index=first_names)
print(loadings.abs().sort_values(ascending=False).head(6).round(3).to_string())
print(f"\ncolumns with a loading above 0.05 in absolute value:"
      f" {int((loadings.abs() > 0.05).sum())} of {len(loadings)}")

**Fourteen** components retain 80% of the variance in these forty-nine
columns, and the SVD of the centred matrix returns the same shares and the same
three counts, because that factorisation is what `PCA` runs. The first component
loads on **16** columns, led by `Other training`, `Plyometric training` and
`Train bf competition`. A model can attribute a result to that component, but not
to one named measurement of the athlete.

__Step 15:__ Does the compressed frame predict as well as the full one? Inside
every training split, on the frame the recipe selected, keep the components that
reach 80% of that split's variance, each reducer counting for itself, then compare against all the features.

In [ ]:
reducers = {"PCA": lambda n: PCA(n_components=n),
            "TruncatedSVD": lambda n: TruncatedSVD(n_components=n,
                                                   random_state=RANDOM_STATE)}

reduction = {"all features": {"features": reference["n"].mean(),
                              "F1": reference["F1"].mean()}}
for name, make in reducers.items():
    run = heldout_f1(recipe, reduce=make)
    reduction[name] = {"features": run["n"].mean(), "F1": run["F1"].mean(),
                       **paired_delta(run["F1"], reference["F1"])}
reduction = pd.DataFrame(reduction).T
print(reduction.to_string(na_rep="", formatters={
    "features": "{:.1f}".format, "F1": "{:.4f}".format,
    "dF1": lambda value: "" if pd.isna(value) else f"{value:+.4f}",
    "SEM": lambda value: "" if pd.isna(value) else f"{value:.4f}",
}))

<div class="alert alert-block alert-warning">

Keeping 80% of each training split's variance takes **14**
components with `PCA` and **15** with `TruncatedSVD`. The full width scores
0.8523 F1, the 14 components score 0.8191 and the 15 score 0.8221, so the model
loses **0.0332 F1** with the first and **0.0302** with the second on **all
twenty** splits.

PCA ranks directions by spread without reading the outcome, so high variance
need not carry predictive signal. This frame also offers little redundancy: 35
of its 49 columns are dummies from 12 independent categoricals.

Reduction suits many redundant or unlabelled columns, including text, images and
sensor arrays, and it supports low-dimensional visualisation. With a few dozen
named measurements, supervised selection can preserve the original labels while
using the outcome.

</div>

# <font color='#E8800A'>Feature selection I: Filter methods</font> <a class="anchor" id="filter"></a>
[Back to TOC](#toc)

Filters rank columns without fitting a predictive model. This
section introduces them one at a time, each with its implementation and its
result: a column that never varies, a column's **relevance** to the outcome, and
a column's **redundancy** with another column.

__Step 16:__ Every technique in this part is judged the same way. `heldout_f1` runs
it through `select` on the twenty splits, and `report` files its rows in one
shared `board` under the technique's name. The board starts with `all features`,
the recipe's own run, which every technique is compared with.

`report` states the result first, in words: the technique's score against the
reference arm, the gap with its standard error, and whether that gap is better,
worse, or inside the split-to-split noise. It then names the columns kept in
most splits. The kept set is a **frequency** rather than one split's list,
because selection is refitted inside every split: a column kept in all twenty
and a column kept in three are different findings.

`report` also files its summary as one row of the `results` DataFrame, so the
techniques can be read side by side.

In [ ]:
board, results = {}, {}
RESULT_FORMAT = {
    "columns": "{:.1f}".format,
    "F1": "{:.4f}".format,
    "dF1": lambda value: "" if pd.isna(value) else f"{value:+.4f}",
    "SEM": lambda value: "" if pd.isna(value) else f"{value:.4f}",
}


def report(name, run):
    """File one technique's rows on the board, then say what it scored and kept.

    The verdict comes first, in words, against the all-features reference. The
    kept set follows as a FREQUENCY over the splits, and only the columns kept
    in most of them are named.
    """
    board[name] = run
    kept = Counter(column for columns in run["kept"] for column in columns)
    row = {"columns": run["n"].mean(), "F1": run["F1"].mean()}
    if ("evaluation" in run
            and run["evaluation"].eq("model-selection validation").all()):
        row["verdict"] = "no unbiased delta"
        line = (f"validation F1 {row['F1']:.4f} with {row['columns']:.1f} columns;"
                " these splits chose this width, so no gap to all features is claimed")
    elif name == "all features":
        line = (f"F1 {row['F1']:.4f} with {row['columns']:.1f} columns: the reference"
                " every other row is compared with")
    else:
        row.update(paired_delta(run["F1"], reference["F1"]))
        line = (f"F1 {row['F1']:.4f} with {row['columns']:.1f} columns, against"
                f" {reference['F1'].mean():.4f} with all features: dF1 {row['dF1']:+.4f}"
                f" +- {row['SEM']:.4f}, better on {row['wins']} splits"
                f" -> {row['verdict'].upper()}")
    results[name] = row
    usual = [f"{c} {n}/{len(splits)}"
             for c, n in kept.most_common() if n > len(splits) / 2]
    rare = sum(n <= len(splits) / 2 for n in kept.values())
    print(name)
    print(textwrap.fill(line, width=84, initial_indent="    ",
                        subsequent_indent="    "))
    print(textwrap.fill("kept in most splits: " + ", ".join(usual), width=84,
                        initial_indent="    ", subsequent_indent="      "))
    if rare:
        print(f"    kept in half the splits or fewer: {rare} more"
              f" column{'' if rare == 1 else 's'}")

__Step 17:__ Start with the reference every filter is compared with: the
recipe's own run, with no column removed.

In [ ]:
report("all features", reference)

Every row below is read against these two numbers. The
reference keeps 48.7 columns on average rather than 49, because the one `No coach`
athlete is held out in six of the twenty splits, so those training rows carry a
single level and its dummy does not exist: one-hot encoding is fitted inside the split, so even the width of the
matrix is a fitted quantity. Its F1 of 0.8523 is the score each selector is compared with.

<div class="alert alert-block alert-info">

**`VarianceThreshold` measures spread, not relevance.** A variance of 0
means the column is constant and can be dropped.

A column with little spread may be uninformative, but variances are
comparable only when their units are. You can set an arbitrary threshold
(e.g. 0.01) to drop columns that vary less than that.

</div>

__Step 18:__ **`VarianceThreshold`.** Look at the spread it would read, on split
0's training matrix exactly as the recipe builds it: the one-hot block, the
measured block, and how many columns have no spread at all.

In [ ]:
variances = pd.Series(first_matrix.var(axis=0), index=first_names)
measured = variances.index.isin(numeric)
print(f"{len(variances)} columns, {int((variances == 0).sum())} constant")
print(f"dummies : variance {variances[~measured].min():.4f} to {variances[~measured].max():.4f}")
print(f"measured: variance {variances[measured].min():.4f} to {variances[measured].max():.4f}")

Nothing is constant, so a threshold of zero has nothing to
remove. The two blocks also sit on different scales for reasons that have nothing
to do with usefulness: a dummy's variance is the prevalence times one minus the
prevalence, which cannot exceed 0.25 and falls to 0.0003 for a level almost
nobody has, while the measured columns have been logged and robust-scaled and run
from 0.1643 to 1.2211. Ranking these 49 columns by spread would therefore rank
the measurements above the dummies by construction.

In [ ]:
report("VarianceThreshold(0)",
       heldout_f1(recipe, select=lambda split: VarianceThreshold(0.0)))

The filter removed nothing, so its row is the reference arm
under a second name and the difference is exactly zero on all twenty splits.
`VarianceThreshold` is a constant-column detector, and this recipe produces no
constant columns. It is useful after a hard filter or a rare-level merge, where a
column can end up constant, and not as a way of ranking.

<div class="alert alert-block alert-info">

**Correlation has two uses in selection.** **Relevance** compares a
column with the outcome: does it carry information about who wins?
**Redundancy** compares two columns with each other: do they record the same fact
twice? A filter needs both, because a ranking by relevance alone cannot see that
two of its top columns are copies of one another.

**The index must match the pair of types.** An encoded dummy asks whether one
level, such as Lisbon, is informative. The source column asks whether `Region` is
informative as a whole.

- For a **measurement against a two-level outcome**, use the point-biserial
  coefficient.
- For a **categorical against a two-level outcome**, use a
  [chi-square test of independence](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chi2_contingency.html)
  and report **Cramér's V**. Unlike the chi-square statistic, V adjusts for row
  count and number of levels, placing the result on [0, 1] beside an |r|.

Encoding changes storage, not variable type. A dummy has a numeric
representation, but numeric representation does not make it a measured variable.
The encoded `SelectKBest` below scores a measured column by
|point-biserial r|, which is monotone-equivalent to `f_classif` for a binary
target, and a 0/1 dummy by Cramér's V (phi in a 2 by 2 table), both on [0, 1],
one encoded column at a time.
`TypedFilter` instead scores each whole source column once and keeps or drops all
of its dummies together.

</div>

__Step 19:__ **Relevance, source columns.** Score every source column against
the outcome with the index its type allows, on split 0's training rows filled by
the recipe. Print the smallest expected count beside each chi-square, because it decides
whether the p-value can be read.

In [ ]:
def cramers_v(column, outcome):
    """Chi-square on a contingency table, rescaled so columns compare.

        Returns the statistic, its p-value, Cramer's V, and the smallest expected
    count. The p-value is an approximation that needs every expected count above
    about 5, so a level almost nobody occupies breaks it. V is still readable as
    an effect size.
    """
    table = pd.crosstab(column, outcome)
    if min(table.shape) < 2:
        # One level present in these rows, so there is no association to
        # measure. `No coach` is True for a single athlete.
        return 0.0, float("nan"), 0.0, float(table.to_numpy().min())
    statistic, p, _, expected = chi2_contingency(table, correction=False)
    v = np.sqrt(statistic / (table.to_numpy().sum() * (min(table.shape) - 1)))
    return statistic, p, v, expected.min()


def typed_relevance(rows, outcome):
    """One comparable number per SOURCE column, each by the index its type allows.

    |point-biserial r| for a measurement, Cramer's V for a categorical. Both are
    on [0, 1], so a fourteen-level column and a duration can be ranked together.
    A measurement is read as the recipe leaves it, because that is the column the
    model will see.
    """
    scores = {}
    for column in numeric:
        values = rows[column].to_numpy(dtype=float)
        if column in log1p_columns:
            values = np.log1p(values)
        scores[column] = abs(
            pointbiserialr(outcome.to_numpy(dtype=int), values).statistic)
    for column in categorical:
        scores[column] = cramers_v(rows[column].astype(str), outcome)[2]
    return pd.Series(scores).sort_values(ascending=False)

In [ ]:
# Split 0's training rows, filled by the recipe's imputers.
plan = [(numeric, recipe["fill"]["numeric"]),
        (categorical, recipe["fill"]["categorical"])]
first_rows, _ = fill_missing(X.iloc[first_index], X.iloc[first_index], plan)
source_scores = typed_relevance(first_rows, first_y)
table = []
for column in source_scores.index:
    if column in numeric:
        table.append({"column": column, "index": "|point-biserial r|",
                      "value": source_scores[column]})
    else:
        statistic, p, v, expected = cramers_v(
            first_rows[column].astype(str), first_y)
        table.append({"column": column, "index": "Cramer's V", "value": v,
                      "levels": first_rows[column].nunique(), "chi2": statistic,
                      "p": p, "min expected": expected})
print(pd.DataFrame(table).round(4).to_string(index=False, na_rep=""))

<div class="alert alert-block alert-success">

Training volume dominates the top of the table. Hours of
training before competition score **0.49** and strength training **0.40** on a
point-biserial |r|. The strongest categorical,
`Past injuries` at V 0.22, still ranks eighth, so a budget of eight source
columns would be spent almost entirely on measurements.

Two rows carry a caveat instead of a finding. `Region` has V 0.08 at p 0.0969, so
its association is both weak and unresolved on 3,200 training rows. `No coach`
has V 0.02 at p 0.2167 with a smallest expected count of 0.40, far below the
counts a chi-square approximation needs, so that p-value should not be read.

</div>

Plot the relevance scores. The heatmap shows the
type-matched association between each source column and the outcome, and each
value names the index that produced it.

In [ ]:
# Each cell names the index it is, because one colour scale carries two of
# them.
relevance_view = source_scores.sort_values().to_frame("association")
relevance_labels = [[f"{value:.2f}  {'|r|' if name in numeric else 'V'}"]
                    for name, value in relevance_view["association"].items()]
fig, ax = plt.subplots(figsize=(5, 9))
sns.heatmap(relevance_view, annot=relevance_labels, fmt="", cmap="Blues",
            vmin=0, vmax=1, cbar=False, ax=ax)
ax.set(title="Relevance to Outcome\n|r| point-biserial, V Cramer's V",
       xlabel="", ylabel="")
fig.tight_layout()
plt.show()

__Step 20:__ **`SelectKBest`, encoded columns.** It ranks the encoded columns one
at a time and keeps the top `k`. Its score function, `encoded_effect_scores`,
gives a dummy its Cramér's V and a measurement its |point-biserial r|, so both
are ranked on the same [0, 1] scale. Try a narrow budget and a wide one.

In [ ]:
def encoded_effect_scores(matrix, outcome):
    """One comparable [0, 1] effect for each encoded column.

    One-hot dummies come first and measured columns come second because that is
    the order `fold_matrices` returns. The explicit binary check prevents a
    transformed measurement from being mistaken for categorical evidence.
    """
    matrix = np.asarray(matrix, dtype=float)
    categorical_width = matrix.shape[1] - len(numeric)
    dummy_block = matrix[:, :categorical_width]
    if not np.isin(dummy_block, [0.0, 1.0]).all():
        raise ValueError("Cramer's V requires the 0/1 one-hot dummy block")

    target = np.asarray(outcome, dtype=int)
    scores = []
    for position in range(matrix.shape[1]):
        values = matrix[:, position]
        if position < categorical_width:
            effect = cramers_v(pd.Series(values), pd.Series(target))[2]
        else:
            effect = abs(pointbiserialr(target, values).statistic)
        scores.append(effect)
    return np.nan_to_num(np.asarray(scores, dtype=float))

In [ ]:
report("SelectKBest(k=10)", heldout_f1(
    recipe, select=lambda split: SelectKBest(encoded_effect_scores, k=10)))
report("SelectKBest(k=25)", heldout_f1(
    recipe, select=lambda split: SelectKBest(encoded_effect_scores, k=25)))

Both budgets lose on every split, by more than ten times the standard
error of the mean gap. Widening from ten columns
to twenty-five recovers about half the gap, from -0.0388 to -0.0219, so the
columns ranked eleventh to twenty-fifth carry something the top ten do not.

The kept sets show where the loss comes from. At k=10 the filter spends eight
slots on measurements and one on `Past injuries`, and its tenth decision moves
between `Plyometric training`, kept in 15 splits, and `Mental preparation`, kept
in 5. Scoring one column at a time cannot notice that several of those eight
measurements record the same training load twice.

__Step 21:__ **`TypedFilter`, source columns.** It asks the relevance question of
the source columns in the table above, keeps the `k` strongest, and keeps or
drops all of a categorical's dummies together. Keep eight source columns.

In [ ]:
class TypedFilter:
    """Keep the k SOURCE columns with the strongest typed relevance.

    A dummy is an encoded column, not an original/source variable: twelve of
    them can represent one categorical. This filter scores `Region` once, from
    its own contingency table, and keeps or drops all of its dummies together.
    It reads the split's own training rows, so nothing held out reaches the
    ranking.
    """

    def __init__(self, k, split_number):
        self.k = k
        self.split_number = split_number

    def fit(self, frame, outcome):
        train_index, _ = splits[self.split_number]
        rows, _ = fill_missing(X.iloc[train_index], X.iloc[train_index], plan)
        best = set(typed_relevance(rows, outcome).head(self.k).index)
        source_of = [next((c for c in categorical if name.startswith(c + "_")), name)
                     for name in frame.columns]
        self.support_ = np.array([source in best for source in source_of])
        return self

    def get_support(self):
        return self.support_

In [ ]:
report("chi-square / |r| on 8 source columns",
       heldout_f1(recipe, select=lambda split: TypedFilter(8, split)))

Scoring whole source columns produces the same eight decisions in every split and the largest gap so far, -0.0417. The filter keeps seven
measurements and `Past injuries`, most of the k=10 set with two fewer columns,
and it still ranks one source column at a time.

__Step 22:__ **Redundancy.** A relevance ranking compares each column only with
the outcome, so it cannot see two columns that record the same fact. Measure the
encoded columns against **each other**, on the split-0 training matrix the
variance check read.

In [ ]:
REDUNDANT = 0.70   # above this, treat a pair as one fact recorded twice


def pair_index(left, right):
    """Name the index a pair of columns is entitled to.

        All three are the same arithmetic on the same two vectors, and they carry
    different names because the types differ: Pearson's r between two
    measurements, the point-biserial coefficient between a measurement and a 0/1
    dummy, phi between two dummies.
    """
    measurements = sum(name in numeric for name in (left, right))
    return {2: "Pearson r", 1: "point-biserial", 0: "phi"}[measurements]

In [ ]:
between = np.abs(np.corrcoef(first_matrix, rowvar=False))
pairs_above = [
    (first_names[i], first_names[j], between[i, j])
    for i, j in zip(*np.triu_indices_from(between, 1))
    if between[i, j] > REDUNDANT
]
print(f"pairs above {REDUNDANT}: {len(pairs_above)} of"
      f" {between.shape[0] * (between.shape[0] - 1) // 2}")
for left, right, value in sorted(pairs_above, key=lambda row: -row[2]):
    print(f"  {left:26s} {right:26s} {value:.4f}  {pair_index(left, right)}")

The heatmap shows the encoded feature pairs that cross the
redundancy threshold. It is not one Pearson correlation matrix: each pair is
named by the index its types allow.

In [ ]:
SHORT_INDEX = {"Pearson r": "r", "point-biserial": "pb", "phi": "phi"}

redundant_names = sorted({name for left, right, _ in pairs_above
                          for name in (left, right)})
redundant_at = [list(first_names).index(name) for name in redundant_names]
redundancy_view = pd.DataFrame(
    between[np.ix_(redundant_at, redundant_at)],
    index=redundant_names, columns=redundant_names,
)
redundancy_labels = [
    [f"{redundancy_view.loc[row, column]:.2f}\n{SHORT_INDEX[pair_index(row, column)]}"
     for column in redundancy_view.columns]
    for row in redundancy_view.index
]
mask = np.triu(np.ones_like(redundancy_view, dtype=bool))
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(redundancy_view, mask=mask, annot=redundancy_labels, fmt="",
            cmap="Oranges", vmin=0, vmax=1, cbar=False, ax=ax)
ax.set(title="Redundancy above 0.70\nr Pearson, pb point-biserial, phi",
       xlabel="", ylabel="")
fig.tight_layout()
plt.show()

Four pairs out of 1,176 cross 0.70, and three of them involve
`Train bf competition`, which repeats `Supplements` at 0.76, `Recovery` at 0.76
and `Strength training` at 0.71, each as a Pearson r between two measurements.
That is one training-volume fact recorded four times, and `SelectKBest(k=10)`
spent four of its ten slots on it.

The fourth pair is different in kind. `Education_High school` and
`Education_Middle school` reach phi 0.72 because they are two levels of one
variable, so an athlete who has one cannot have the other. Dummies from the same
source exclude one another by construction, and the size of that relation is a
fact about one-hot encoding rather than about education.

__Step 23:__ **`CorrelationFilter`, relevance and redundancy together.** It walks
the same relevance ranking as `SelectKBest` and skips a candidate that is
redundant with a column already kept. At the same `k` the skip is the only
difference between the two, so their paired gap is what the rule is worth.

In [ ]:
class CorrelationFilter:
    """Rank by |r| with the outcome, then walk the ranking and skip repeats.

    A candidate is skipped when it correlates above `redundancy` with a column
    already kept: the pair is one fact recorded twice, and the second copy costs
    a slot the filter could have spent elsewhere. With `redundancy=None` this is
    the encoded `SelectKBest` baseline: |point-biserial r| for measurements and
    Cramer's V for dummy flags. Phi and point-biserial r have the same magnitude
    for a binary dummy, but naming V preserves the variable's categorical type.
    With `k=None` there is no budget to exhaust, so the redundancy rule runs
    alone and every column that repeats nothing already kept survives.
    """

    def __init__(self, k, redundancy=None):
        self.k = k
        self.redundancy = redundancy

    def fit(self, frame, outcome):
        matrix = frame.to_numpy(dtype=float)
        # A constant column correlates with nothing, and the arithmetic divides
        # by its zero spread. Both are read as "no relation".
        with np.errstate(invalid="ignore", divide="ignore"):
            relevance = encoded_effect_scores(matrix, outcome)
            between = np.nan_to_num(np.abs(np.corrcoef(matrix, rowvar=False)))
        keep = []
        for candidate in np.argsort(relevance)[::-1]:
            if len(keep) == self.k:
                break
            if self.redundancy is not None and any(
                between[candidate, chosen] > self.redundancy for chosen in keep
            ):
                continue
            keep.append(int(candidate))
        self.support_ = np.zeros(matrix.shape[1], dtype=bool)
        self.support_[keep] = True
        return self

    def get_support(self):
        return self.support_

In [ ]:
report("correlation k=10, redundancy dropped", heldout_f1(
    recipe, select=lambda split: CorrelationFilter(10, redundancy=REDUNDANT)))

skip = paired_delta(board["correlation k=10, redundancy dropped"]["F1"],
                    board["SelectKBest(k=10)"]["F1"])
print(f"redundancy rule vs SelectKBest(k=10): dF1 {skip['dF1']:+.4f}"
      f" +- {skip['SEM']:.4f}, better on {skip['wins']} splits"
      f" -> {skip['verdict'].upper()}")

Skipping the repeats changed which ten columns survive.
`Recovery`, `Supplements` and `Strength training` leave the set, and `Mental
preparation`, `Other training` and `Sand training` take the freed slots.

It did not pay for itself. Against `SelectKBest(k=10)`, which walks the same
ranking with the skip switched off, the paired difference is -0.0025 +- 0.0013
and the redundancy rule is ahead on 8 of the 20 splits. At a fixed budget of ten,
the columns the skip buys are worth about what the copies it removed were worth.

__Step 24:__ Read the filters side by side. Every row is compared with `all features`.

In [ ]:
print(pd.DataFrame(results).T.to_string(formatters=RESULT_FORMAT, na_rep=""))

<div class="alert alert-block alert-success">

Every filter row that removes a column loses to the full matrix, and
the two rows that lose least are the two widest. `VarianceThreshold` removes nothing and
matches the reference exactly, k=25 gives up 0.0219, and the three rows at eight
to ten columns give up about 0.04. Width explains most of the spread in this
table; method explains the rest.

The typed filter needs eight quantities per athlete instead of twenty-six, at a
cost of 0.0417 F1.

</div>

# <font color='#E8800A'>Feature selection II: Wrapper methods</font> <a class="anchor" id="wrapper"></a>
[Back to TOC](#toc)

A wrapper judges features through a fitted model, so it can detect
joint effects that a univariate filter misses. Some wrappers cross-validate
internally, which adds another fit boundary.

__Step 25:__ **Exercise.** Inspect the three wrapper selectors' signatures for a
`cv` parameter. Classify which selectors run an internal validation loop and
which one applies a fixed feature budget without one.

In [ ]:
# 1. report each selector's cv default, or that it has none: a selector with
#    no cv parameter applies the feature budget you hand it
for cls in (RFE, RFECV, SequentialFeatureSelector):
    parameters = ...  # <-- CODE HERE
    if "cv" in parameters:
        # 2. `cv=None` is not "no cross-validation": the docstring says what the
        #    class does with it
        default = ...  # <-- CODE HERE
        folds = ...  # <-- CODE HERE
        print(f"{cls.__name__:26s} cv default = {default!r}, so {folds}-fold")
    else:
        print(f"{cls.__name__:26s} NO cv PARAMETER: fixed-budget fit")

<div class="alert alert-block alert-warning">

**Two wrappers cross-validate internally.** `RFECV` defaults to
five folds through `cv=None`, and `SequentialFeatureSelector` defaults to
`cv=5`. Preprocessing the outer training rows once before either selector leaks
information from each internal validation fold into its internal training fold.

The safe remedy is to fit preprocessing separately on each internal training
fold, transform its validation fold, and use an explicit `PredefinedSplit` to
keep those roles visible. The full implementation is deferred until nested
validation is introduced. This notebook stops at the warning and benchmarks
`RFE`, which has no internal cross-validation and takes its feature budget as an
argument, so the width is chosen outside the selector.

</div>

__Step 26:__ Search the existing twenty train/validation splits for a width.
`RFE` runs at 5, 10, 15 and so on up to the narrowest split's width, eliminating
one column per step as the combinations below do. The single width with the
highest mean validation F1 wins, and an exact tie prefers the smaller model.

Its score is **model-selection validation**, not an untouched test estimate,
because the same validation results both choose `k` and describe it. Unbiased
nested evaluation is deferred until the course introduces that boundary.

In [ ]:
common_width = int(reference["n"].min())
candidate_counts = list(range(5, common_width + 1, 5))
rfe_runs = {k: heldout_f1(recipe, select=lambda split: RFE(
                LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
                n_features_to_select=k))
            for k in candidate_counts}

rfe_curve = pd.DataFrame({
    "k": candidate_counts,
    "mean validation F1": [rfe_runs[k]["F1"].mean() for k in candidate_counts],
    "validation F1 SEM": [rfe_runs[k]["F1"].sem() for k in candidate_counts],
})
best_mean = rfe_curve["mean validation F1"].max()
chosen_k = int(rfe_curve.loc[
    rfe_curve["mean validation F1"] == best_mean, "k"
].min())
rfe_curve["selected"] = rfe_curve["k"].eq(chosen_k)

print(rfe_curve.round(4).to_string(index=False))
print(f"selected k = {chosen_k} by mean validation F1; exact ties prefer smaller k")

Only the chosen width goes on the board, under one name.

In [ ]:
report("RFE(validation-selected)",
       rfe_runs[chosen_k].assign(evaluation="model-selection validation"))

__Step 27:__ Plot the search: the mean validation F1 at each width, with its
standard error, and the width the rule landed on.

In [ ]:
selected_point = rfe_curve.loc[rfe_curve["selected"]].iloc[0]
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.errorbar(
    rfe_curve["k"], rfe_curve["mean validation F1"],
    yerr=rfe_curve["validation F1 SEM"], marker="o", capsize=3,
    color=PLOT_ORANGE, label="validation",
)
ax.axvline(chosen_k, color="0.45", linestyle=":", linewidth=1.5)
ax.scatter(
    [chosen_k], [selected_point["mean validation F1"]],
    marker="*", s=180, color=PLOT_ORANGE, edgecolor="black", zorder=5,
)
ax.annotate(
    f"selected k={chosen_k}, validation F1={selected_point['mean validation F1']:.4f}",
    xy=(chosen_k, selected_point["mean validation F1"]),
    xytext=(8, 10), textcoords="offset points",
)
ax.set_xlabel("number of selected features")
ax.set_ylabel("mean F1")
ax.set_title("RFE selection curve on the validation splits")
ax.legend()
fig.tight_layout()
plt.show()

The winning mean is optimistic, because it is the maximum over the candidates
on the validation splits that chose it. `RFE` supplies the ranking and those
splits choose the width, so the board holds one row for that choice rather than
one per tried `k`.

__Step 28:__ The table again, with the wrapper rows in it.

In [ ]:
print(pd.DataFrame(results).T.to_string(formatters=RESULT_FORMAT, na_rep=""))

# <font color='#E8800A'>Feature selection III: Embedded methods</font> <a class="anchor" id="embedded"></a>
[Back to TOC](#toc)

An embedded method selects during model fitting. Here,
L1-penalised logistic regression can shrink coefficients to zero, while a tree
supplies impurity-based importances. A fixed penalty keeps hyperparameter search outside this comparison.

<div class="alert alert-block alert-info">

**A coefficient or an importance is a score, not a decision.** A
fitted L1 model gives every column a coefficient, and a fitted tree gives every
column an impurity importance. Neither says which columns to keep.
[`SelectFromModel`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectFromModel.html)
makes that decision: it fits the model, reads `coef_` or `feature_importances_`,
and keeps the columns whose absolute value reaches `threshold`.

The threshold therefore carries the decision, and each signal needs its own:

- **L1.** The penalty itself sets weak coefficients to exactly zero, so
  `threshold=1e-8` keeps the columns the penalty left alive. The penalty strength
  `C` decides how many survive.
- **Tree importance.** Importances sum to 1 and have no natural cut, so
  `threshold="median"` keeps the more important half by construction. That width
  is chosen in advance; it is not a finding about the data.

</div>

__Step 29:__ **L1 embedded.** Measure the penalised model, then look at what it decided. The figure averages each column's coefficient over the splits; the whisker is the spread across splits, and a grey bar
marks a column the penalty removed in most of them.

In [ ]:
# `l1_ratio=1` IS the lasso penalty. scikit-learn 1.8 deprecated the older
# spelling of the same model, `penalty='l1'`, and removes it in 1.10;
# `l1_ratio=0` would be ridge, and a value between the two is elastic net.
report("L1 embedded", heldout_f1(recipe, select=lambda split: SelectFromModel(
    LogisticRegression(l1_ratio=1, solver="liblinear", C=0.1, max_iter=1000,
                       random_state=RANDOM_STATE),
    threshold=1e-8)))

In [ ]:
def per_split(name, read):
    """`read` applied to every split's fitted selector, one row per split.

    Columns are matched by NAME: a level absent from a training split has no
    column there, so that split leaves it missing rather than zero.
    """
    return pd.DataFrame([pd.Series(read(fitted), index=fitted.feature_names_in_)
                         for fitted in board[name]["selector"]])


coefficients = per_split("L1 embedded", lambda fitted: fitted.estimator_.coef_[0])
kept_share = per_split("L1 embedded", lambda fitted: fitted.get_support()).astype(float).mean()
l1_order = coefficients.mean().abs().sort_values().index
fig, ax = plt.subplots(figsize=(7, 11))
ax.barh(l1_order, coefficients.mean()[l1_order], xerr=coefficients.std()[l1_order],
        color=np.where(kept_share[l1_order] > 0.5, PLOT_BLUE, "0.75"))
ax.axvline(0, color="0.3", linewidth=0.8)
ax.set(title=f"L1 coefficients, mean of {len(coefficients)} splits:"
             f" {int((kept_share > 0.5).sum())} of {len(l1_order)} kept in most",
       xlabel="coefficient (whisker: spread across splits)", ylabel="")
fig.tight_layout()
plt.show()
print(coefficients.mean()[l1_order[::-1]].head(3).round(2).to_string())

27 of the 49 columns survive in most splits, and the largest coefficients vary
little between splits: `Competition_Local Match` sits near -2.0, and the
competition levels, `Mental preparation` and `Train bf competition` near +1.
`Recovery` and `Sport-specific training` rank fourth and sixth in the relevance
table, and L1 drops both in most splits because `Train bf competition` already
carries their information.

This is the first technique on the board inside the split-to-split noise: 26.1
columns, F1 0.8511, a gap of -0.0012 +- 0.0006, so nearly half the columns go at
no measurable cost.

__Step 30:__ **Tree importance embedded.** Measure a tree with
`threshold="median"`, then average its importances the same way. The dashed line
is the median cut, averaged over the splits.

In [ ]:
report("DT importance embedded", heldout_f1(
    recipe, select=lambda split: SelectFromModel(
        DecisionTreeClassifier(random_state=RANDOM_STATE), threshold="median")))

In [ ]:
importances = per_split("DT importance embedded",
                        lambda fitted: fitted.estimator_.feature_importances_)
kept_share = per_split("DT importance embedded", lambda fitted: fitted.get_support()).astype(float).mean()
cut = np.mean([fitted.threshold_ for fitted in board["DT importance embedded"]["selector"]])
print(importances.mean().sort_values(ascending=False).head(2).round(2).to_string())
tree_order = importances.mean().sort_values().index
fig, ax = plt.subplots(figsize=(7, 11))
ax.barh(tree_order, importances.mean()[tree_order], xerr=importances.std()[tree_order],
        color=np.where(kept_share[tree_order] > 0.5, PLOT_ORANGE, "0.75"))
ax.axvline(cut, color="0.3", linestyle="--", linewidth=1)
ax.set(title=f"Tree importances, mean of {len(importances)} splits",
       xlabel="impurity importance\n(whisker: spread across splits; dashed: mean median cut)",
       ylabel="")
fig.tight_layout()
plt.show()
print(f"mean median cut {cut:.4f}; kept in most splits:"
      f" {int((kept_share > 0.5).sum())} of {len(tree_order)}")

Averaged over the splits, `Train bf competition` takes about
0.29 of the importance and `Cardiovascular training` about 0.13, and the other 47
columns share the rest. The mean median cut, 0.0063, falls inside that flat tail,
so `threshold="median"` separates columns whose importances differ in the third
decimal place, and the grey bars just below the line are the ones it happens to
drop.

The two embedded methods keep different halves. The tree keeps `Recovery`,
`Sport-specific training` and `Sex_M` in every split, all three of which the L1
model drops, and it drops `Mental preparation` and `Competition_Olympic Games`,
which L1 weights among its largest. The tree's half costs -0.0105 +- 0.0016 and
is worse on all twenty splits, against -0.0012 for L1 at a similar width.

__Step 31:__ The table again, with the embedded rows in it.

In [ ]:
print(pd.DataFrame(results).T.to_string(formatters=RESULT_FORMAT, na_rep=""))

# <font color='#E8800A'>Combining strategies</font> <a class="anchor" id="combined"></a>
[Back to TOC](#toc)

Selectors can run **in sequence**, each narrowing the next one's
input, or in parallel as a **vote**. Both combinations below use the same three
techniques with the same settings, so the only thing
that differs between them is the rule that combines the techniques.

A combination is what this notebook exports, because the algorithm is not chosen
yet. Each family reads features through its own instrument: a filter through one
column at a time, a wrapper through one fitted estimator, an embedded method
through the model it is part of. A set that two or three of those readings agree
on leans less on any one of them, and the weeks ahead fit other algorithms on
it. That is the argument for combining
rather than a result this notebook measures: measuring it would take a second
algorithm, which is a later week's work.

So the question here is not which technique wins. It is which rule for combining
them to export.

<div class="alert alert-block alert-info">

A **sequence** forms a funnel: each technique sees only the columns
the previous one kept, so the later ones fit on fewer columns, but an early
rejection is final and can erase a useful joint effect.

A **vote** fits every technique on the same frame and keeps the columns with
enough support. One technique cannot decide the result, but shared blind spots
survive the vote, and every technique still pays its full fitting cost.

</div>

__Step 32:__ Write both combinations as selectors. Each records, in `decisions_`,
which technique kept each column and whether the combination kept it, so the fitted copies from every split can be read together rather than one at a time. Both
can also describe the configuration they were built from, because the strategy
this notebook ends up selecting has to reach the log in a form a later week can
rebuild.

In [ ]:
def describe(selector):
    """The constructor call that built a selector, as text for the log.

    A scikit-learn estimator keeps its parameters as plain attributes and marks
    everything it learned with a trailing underscore, so dropping those leaves
    the configuration it was handed.
    """
    def literal(value):
        if hasattr(value, "get_params"):
            return describe(value)
        if callable(value):
            return value.__name__
        return repr(value)

    parameters = (selector.get_params(deep=False)
                  if hasattr(selector, "get_params") else vars(selector))
    arguments = ", ".join(f"{name}={literal(value)}"
                          for name, value in parameters.items()
                          if not name.endswith("_"))
    return f"{type(selector).__name__}({arguments})"


class Sequence:
    """Apply selectors one after another, each on what the previous one left.

    The mask is composed rather than recomputed: step two is fitted on the
    columns step one kept, and its support is written back into their positions.
    `decisions_` says, for every column, whether it was still held after each
    stage, so the counts can only shrink from one stage to the next.
    """

    def __init__(self, *stages):
        self.stages = stages

    def fit(self, frame, outcome):
        self.feature_names_in_ = frame.columns.to_numpy()
        support = np.ones(frame.shape[1], dtype=bool)
        decisions = {}
        for name, stage in self.stages:
            survivors = frame.loc[:, support]
            kept = stage.fit(survivors, outcome).get_support()
            support[np.flatnonzero(support)] = kept
            decisions[name] = support.copy()
        self.support_ = support
        self.decisions_ = pd.DataFrame(decisions, index=frame.columns)
        self.decisions_["kept"] = support
        return self

    def get_support(self):
        return self.support_

    def spec(self):
        """The stages in order, each as the constructor call that built it."""
        return {"kind": "sequence",
                "stages": [{"stage": name, "selector": describe(stage)}
                           for name, stage in self.stages]}


class Vote:
    """Keep the columns that at least `minimum` of the members select.

    Every member is fitted on the same training rows, so the vote is over
    methods rather than over rows, and a column kept by one member alone is not
    kept by the vote. `decisions_` records which member kept which column.
    """

    def __init__(self, members, minimum):
        self.members = members
        self.minimum = minimum

    def fit(self, frame, outcome):
        self.feature_names_in_ = frame.columns.to_numpy()
        self.decisions_ = pd.DataFrame(
            {name: member.fit(frame, outcome).get_support()
             for name, member in self.members.items()},
            index=frame.columns,
        )
        self.support_ = (self.decisions_.sum(axis=1) >= self.minimum).to_numpy()
        self.decisions_["kept"] = self.support_
        return self

    def get_support(self):
        return self.support_

    def spec(self):
        """The threshold and the members, each as the call that built it."""
        return {"kind": "vote", "minimum": self.minimum,
                "members": {name: describe(member)
                            for name, member in self.members.items()}}


def majority_table(name):
    """Keep counts over the twenty splits, for the columns kept in most of them.

    Each entry counts the splits in which that technique kept the column, and
    the last column counts the splits in which the combination kept it. A
    column the combination kept in half the splits or fewer is left out.
    """
    runs = board[name]["selector"]
    counts = pd.concat([run.decisions_ for run in runs]).groupby(level=0).sum()
    shown = counts[counts["kept"] > len(runs) / 2].sort_values(
        list(counts.columns[::-1]), ascending=False)
    print(f"{len(shown)} of {len(counts)} columns kept in most of the"
          f" {len(runs)} splits")
    print((shown.astype(int).astype(str) + f"/{len(runs)}").to_string())

__Step 33:__ **The three shared techniques**, one per family: the correlation
filter keeps the 35 most relevant columns that repeat nothing already kept, `RFE`
keeps the width the validation curve selected, and the L1 model keeps the
coefficients its penalty leaves alive. Every call builds fresh, unfitted copies,
so nothing fitted in one split reaches another. These settings were chosen on
this board so that neither combination is handicapped, which gives both the
optimism the curve's maximum already carries.

In [ ]:
def shared_techniques():
    """One technique per family, in funnel order: filter, wrapper, embedded."""
    return [
        ("filter", CorrelationFilter(35, redundancy=REDUNDANT)),
        ("wrapper", RFE(LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
                        n_features_to_select=chosen_k)),
        ("embedded", SelectFromModel(
            LogisticRegression(l1_ratio=1, solver="liblinear", C=0.1,
                               max_iter=1000, random_state=RANDOM_STATE),
            threshold=1e-8)),
    ]

__Step 34:__ **A sequence.** Run the three in that order, each on the columns the
previous one kept.

In [ ]:
report("sequence: filter -> wrapper -> embedded", heldout_f1(
    recipe, select=lambda split: Sequence(*shared_techniques())))

__Step 35:__ Read the sequence over every split. The table lists only the
columns the sequence kept in most splits, and each entry counts the splits in
which that stage still held the column.

In [ ]:
majority_table("sequence: filter -> wrapper -> embedded")

The funnel loses. 21.8 columns give F1 0.8492, a gap of
-0.0031 +- 0.0012 that lies outside the noise, and the sequence is ahead on only
7 of the 20 splits. Most rows hold their count through the filter and the
wrapper, and the L1 stage takes the largest bites: `Sport-specific training`
falls from 20 splits to 14, and `Age group_35-55` from 19 to 14.

The table does not show what the filter removed first. `Strength training`
repeats `Train bf competition`, so the redundancy rule drops it in 18 of the 20
splits and neither later stage sees it, although both keep it in every split
when they are given it.

__Step 36:__ **A vote.** Fit the same three on the full training matrix, and keep
a column when at least two of them keep it.

In [ ]:
report("vote: 2 of 3 families", heldout_f1(
    recipe, select=lambda split: Vote(dict(shared_techniques()), minimum=2)))

__Step 37:__ Read the vote the same way. Each entry now counts the splits in which
that technique, on its own, kept the column, and the last column counts the
splits in which at least two of them agreed.

In [ ]:
majority_table("vote: 2 of 3 families")

The vote does not lose: 30.2 columns give F1 0.8525 against
0.8523 with all features, a gap of +0.0001 +- 0.0007, ahead on 11 of the 20
splits and well inside the noise. It keeps more than the sequence because no
single technique's rejection is final. `Strength training`, which the filter
drops in 18 splits, survives in all twenty because the wrapper and the L1 model
both keep it, and `Athlete score` and `Sand training`, which never pass the
wrapper, survive on the filter and the L1 model instead.

A sequence lets any stage veto a column, while a vote asks two of three to
agree. Here that difference separates a measurable loss from none, at the cost of
8.4 more columns.

__Step 38:__ The table again, with the combinations in it: every technique and
every attempt.

In [ ]:
print(pd.DataFrame(results).T.to_string(formatters=RESULT_FORMAT, na_rep=""))

<div class="alert alert-block alert-success">

Across every technique and attempt, three rows avoid a
measurable loss: `VarianceThreshold`, which removes nothing, L1 on its own at 26.1
columns, and the vote at 30.2. Every fixed-budget filter is worse, the tree's
median cut is worse, and the sequence is worse. The `RFE` row carries no verdict,
because its width was chosen on these splits.

The rows that keep up with all features keep more than half of its 48.7
columns, and every row narrower than 22 columns loses.

</div>

# <font color='#E8800A'>So what is selection for?</font> <a class="anchor" id="synthesis"></a>
[Back to TOC](#toc)

Selection did not improve prediction; what it changed is the width of the
model.

__Step 39:__ **Exercise.** Collect the rows every section above wrote into
`board`: the mean number of features, the mean held-out F1 and its across-split
standard error. The plot beneath it is given.

In [ ]:
summary = pd.DataFrame({
    name: {
        # 1. the mean number of features this technique kept
        "n": ...,  # <-- CODE HERE
        # 2. the mean held-out F1 across splits
        "F1": ...,  # <-- CODE HERE
        # 3. and the error on that mean: the split-to-split spread divided by
        #    the root of the split count, which is what `.sem()` computes
        "F1 sem": ...,  # <-- CODE HERE
    }
    for name, run in board.items()
}).T
# 4. one row per technique, best F1 first
summary = ...  # <-- CODE HERE
print(summary.round(4).to_string())

# Given: the table as a figure.
plt.figure(figsize=(7.5, 5))
plt.errorbar(
    summary["F1"], np.arange(len(summary)),
    xerr=2 * summary["F1 sem"],
    fmt="o", color=PLOT_BLUE, capsize=4,
)
plt.yticks(np.arange(len(summary)), summary.index)
plt.gca().invert_yaxis()
plt.xlabel("validation F1  (RFE maximum is selection-biased; bars are 2 x SEM)")
plt.title("The fixed-budget filters are the rows clearly apart")
plt.tight_layout()
plt.show()

__Step 40:__ **Exercise.** In two sentences, explain what an RFE ranking can see
that a univariate filter cannot, then explain why the maximum point on the
validation curve is not an unbiased performance estimate.

In [ ]:
# Write your answer below. No Python is required.
#


__Step 41:__ **The decisions this notebook exports.** Record both decisions in one
cell, then write the log once: the encoder and scaler combination, with the
scores of the sixteen combinations it was chosen from, and the selection
strategy. The candidates are the two combinations, for the reason the combining
section gives, and the one with the higher mean paired F1 gap to all features
wins.

In [ ]:
# 1. the encoder + scaler combination chosen at the start, as ONE decision,
#    with the scores of all sixteen combinations it was chosen from
recipe_log.record(
    f"all {len(categorical) + len(numeric)} feature columns",
    f"selected the {selected_encoding} + {selected_scaler} combination: {selected_encoding}"
    f" encoding and {selected_scaler} scaling, both fitted inside each training split",
    "highest mean held-out F1 among the sixteen combinations of the four encoders"
    " and the four Week 3 scalers",
    len(X),
    carries={
        "combination": f"{selected_encoding} + {selected_scaler}",
        "encoding": selected_encoding,
        "scaler": selected_scaler,
        "drop": "first" if selected_encoding == "one-hot" else None,
        "categorical": list(categorical),
        "numeric": list(numeric),
        "fitted_per_split": True,
        "evaluation": "20 repeated 80/20 holdouts",
        "metric": "held-out F1",
        "candidates": {
            f"{encoding_name} + {scaler_name}": {
                metric: float(value) for metric, value in row.items()
            }
            for (encoding_name, scaler_name), row in encoding_board.iterrows()
        },
    },
)

# 2. the selection strategy: the better of the two combinations
candidates = {
    name: board[name]["F1"] - board["all features"]["F1"]
    for name in ("sequence: filter -> wrapper -> embedded", "vote: 2 of 3 families")
}
best = max(candidates, key=lambda name: candidates[name].mean())

recipe_log.record(
    f"{len(first_names)} encoded columns",
    f"select features with the {best} strategy, refitted inside each training split",
    "smaller paired F1 gap to the all-features arm than the other combination;"
    " a combination is used so that no single technique decides the retained set",
    len(X),
    carries={
        "strategy": board[best]["selector"][0].spec(),
        "fitted_per_split": True,
        "evaluation": "20 repeated 80/20 holdouts",
        "metric": "paired dF1 against the all-features arm",
        "candidates": {name: {"dF1": float(gap.mean()),
                              "n": float(board[name]["n"].mean())}
                       for name, gap in candidates.items()},
        "applied": "from Week 5 onward",
    },
)

# 3. one write, with both of this week's decisions in it
recipe_log.to_json("../../logs/week_04_feature_work_classification_log.json")

print(f"logged: {selected_encoding} + {selected_scaler}, then {best}")
print(f"{len(recipe_log)} decisions written to"
      " ../../logs/week_04_feature_work_classification_log.json")

`carries["strategy"]` holds the vote's threshold and each of its three
techniques as the call that built it, so the next notebook rebuilds the vote and
refits it inside its own splits. A list of the columns kept here would carry a
decision made on these 3,200 training rows into rows it never saw.

The vote won, +0.0001 against -0.0031 for the sequence, and it is the only
combination that does not lose to all features: about 30 columns instead of 49
at no measurable cost. A finer search is not worth its cost here, because it
would add fits for differences smaller than their own standard errors.

# <font color='#E8800A'>Key takeaways</font> <a class="anchor" id="takeaways"></a>
[Back to TOC](#toc)

1. **Declare the regime before measuring.** The comparison, split,
   and reference floor must be fixed before reading results.
2. **Reduction keeps variance, not signal.** Fourteen components retain 80% of
   the variance but lose 0.0332 F1 on every split because PCA ignores the target.
3. **Match the index to the types.** Use point-biserial correlation for a
   measurement against this outcome and Cramér's V for a categorical. A
   chi-square p-value is interpretable only when its expected counts permit it.
4. **Measure redundancy separately from relevance.** A univariate filter never
   compares features with one another.
5. **Fit every learned step inside each training split.** Selection reads the
   target, so a selector fitted before the split would choose its columns with
   the same held-out outcomes it is then scored on.
6. **Export a combination, not the best single row.** The algorithm is not
   chosen yet, and each family reads features through its own instrument, so a
   set two of three families keep depends less on any one of them.
7. **Carry the strategy, not the columns.** The retained set belongs to the rows
   it was fitted on. What the log exports is the rule that produced it, so the
   next notebook refits that rule inside its own splits.

### Apply this method

Build and test transformations before selection. Compare filters, wrappers and
embedded methods on the same splits against the same reference, and report each
paired effect with its SEM, selected width, and fit boundary.

# <font color='#E8800A'>References</font> <a class="anchor" id="references"></a>
[Back to TOC](#toc)

- Guyon, I. & Elisseeff, A. (2003). An introduction to variable and feature selection. *JMLR*, 3, 1157-1182.
- Kuhn, M. & Johnson, K. (2019). *Feature Engineering and Selection: A Practical Approach for Predictive Models*. CRC Press, § 1.4 (augment the predictor set, then filter the enhanced set).
- Kaufman, S., Rosset, S. & Perlich, C. (2012). Leakage in data mining: formulation, detection, and avoidance. *ACM TKDD*, 6(4).
- Jolliffe, I.T. & Cadima, J. (2016). Principal component analysis: a review and recent developments. *Phil. Trans. R. Soc. A*, 374(2065).
- scikit-learn User Guide, [Feature selection](https://scikit-learn.org/stable/modules/feature_selection.html) · [`RFE`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.RFE.html) · [`RFECV`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.RFECV.html) · [`SequentialFeatureSelector`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SequentialFeatureSelector.html) · [`SelectKBest`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectKBest.html) · [`VarianceThreshold`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.VarianceThreshold.html) · [`PredefinedSplit`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.PredefinedSplit.html).
- scikit-learn User Guide, [Common pitfalls: data leakage](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage-during-pre-processing).
- numpy documentation, [`numpy.linalg.svd`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.svd.html).